In [ ]:
# Cell 1: imports, device, seed

import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

# 统一随机性
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# Cell 2: define LSTM-based character LM

class LSTMCharModel(nn.Module):
    """
    简单的字符级 LSTM 语言模型：
    - token embedding
    - 多层 LSTM
    - LayerNorm + Linear 输出到 vocab
    接口和 TinyGPT 类似：forward(idx, targets=None) -> (logits, loss)
    """
    def __init__(self, vocab_size, d_model, n_layers, block_size, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.n_layers = n_layers
        self.block_size = block_size

        # token embedding
        self.token_emb = nn.Embedding(vocab_size, d_model)

        # 多层 LSTM；batch_first=True 方便 (B, T, D)
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=d_model,
            num_layers=n_layers,
            batch_first=True,
        )

        self.dropout = nn.Dropout(dropout)
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        """
        idx: (B, T) long
        targets: (B, T) long, 可选
        """
        B, T = idx.size()
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} > block_size {self.block_size}")

        # (B, T, D)
        x = self.token_emb(idx)
        # (B, T, D)
        y, _ = self.lstm(x)
        y = self.dropout(y)
        y = self.ln_f(y)
        # (B, T, vocab)
        logits = self.head(y)

        loss = None
        if targets is not None:
            # 展平后做 cross-entropy
            loss = F.cross_entropy(
                logits.view(-1, self.vocab_size),
                targets.view(-1),
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200):
        """
        简单的按 token 生成接口（和 TinyGPT 类似，只是内部用 LSTM）
        """
        self.eval()
        B = idx.size(0)

        # 初始化隐藏状态为 0
        h = torch.zeros(self.n_layers, B, self.d_model, device=idx.device)
        c = torch.zeros(self.n_layers, B, self.d_model, device=idx.device)

        for _ in range(max_new_tokens):
            # 只取最近 block_size 长度
            idx_cond = idx[:, -self.block_size:]
            x = self.token_emb(idx_cond)           # (B, T, D)
            y, (h, c) = self.lstm(x, (h, c))       # (B, T, D)
            y = self.ln_f(y)
            logits = self.head(y)                  # (B, T, vocab)
            logits_last = logits[:, -1, :]         # 只看最后一个位置
            probs = torch.softmax(logits_last, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat([idx, idx_next], dim=1)

        return idx


In [ ]:
# Cell 3: simple sanity check (optional)

if __name__ == "__main__":
    vocab_size_demo = 50
    d_model_demo = 128
    n_layers_demo = 2
    block_size_demo = 64

    model_demo = LSTMCharModel(
        vocab_size=vocab_size_demo,
        d_model=d_model_demo,
        n_layers=n_layers_demo,
        block_size=block_size_demo,
        dropout=0.1,
    ).to(device)

    x_demo = torch.randint(0, vocab_size_demo, (4, 32), device=device)
    y_demo = torch.randint(0, vocab_size_demo, (4, 32), device=device)

    logits_demo, loss_demo = model_demo(x_demo, y_demo)
    print("logits shape:", logits_demo.shape)
    print("loss:", loss_demo.item())
